# Project 5 — FX Risk Hedging for EUR/PLN and USD/PLN for XTB S.A.
## Enterprise Risk Management

**Context.** XTB S.A. is a Warsaw Stock Exchange (GPW)-listed broker whose revenues and operating costs have
significant FX exposure (foreign branches, EUR/USD settlements, CFD volume on FX).
In `prez3` we showed that a synthetic portfolio mirroring the revenue structure
(50% EUR, 30% USD, 20% GBP) has high component correlation — **hedging is necessary**,
not just diversification.

**Objectives:**
1. Propose a hedging strategy on the **external regulated market** (GPW contracts).
2. Explain why **CFDs on XTB's own platform do not constitute a hedge** (net zero).
3. Determine **hedge ratio**, KDPW margin (effective leverage), **Delta**, and **VaR**.

**Convention:** daily loss reported as a positive value; VaR at the 99% level (1 day).

---
## 1. FX Exposure of XTB S.A.

Based on the 2024 standalone financial statements and the revenue structure from `prez3`,
we adopt the following operational assumptions:

| Exposure source | Description | Currency |
|---|---|---|
| Foreign branch revenues | CFD/stock commissions and spreads | EUR, USD |
| Inter-branch settlements | Transfer pricing, IT, marketing | EUR |
| Provisions for negative client balances | Exposure to FX CFD moves | USD, EUR |
| Cash assets in foreign currencies | Cash in the brokerage account | EUR, USD |

**Net notional exposure** (long foreign-currency position, to be hedged):

| Pair | Net exposure | Rationale |
|---|---:|---|
| **EUR/PLN** | **50 mln EUR** | ~50% of revenue stream (Europe, EUR-denominated CFD) |
| **USD/PLN** | **30 mln USD** | ~30% of revenues (USA, commodities, US indices) |

Exposure means that **PLN weakening** (EUR/USD appreciation vs. PLN) increases the
balance-sheet value of assets in PLN, while **PLN strengthening** generates a transactional loss.
Hedging strategy: take a **short** position in derivative instruments (sell contracts).

---
## 2. Actual Derivative Instruments

### 2a. GPW Futures Contracts (primary hedging instrument)

On the GPW Main Market, **currency futures contracts** are listed, settled
cash against the NBP fixing ([GPW specification](https://www.gpw.pl/kontrakty-terminowe-na-kursy-walut)):

| Parameter | **FEUR** (EUR/PLN) | **FUSD** (USD/PLN) |
|---|---|---|
| Underlying instrument | EUR/PLN rate (NBP) | USD/PLN rate (NBP) |
| Contract size | **1 000 EUR** | **1 000 USD** |
| Quotation | PLN per 1 EUR/USD (tick 0.0001) | ibid. |
| Delivery months | 3 nearest + March/June/September/December | ibid. |
| Settlement | Cash in PLN, NBP fixing on expiry date | ibid. |
| Initial margin | ~4–6% of notional value (KDPW) | ibid. |

**Benefits for XTB:** regulated market, no counterparty risk (KDPW), transparent NBP fixing,
low roll cost on a quarterly schedule.

### 2b. Leverage on GPW Contracts — Do Not Confuse with CFD Leverage

On GPW futures contracts, **retail leverage of 1:30** familiar from the brokerage
platform does not apply. Instead, an **initial margin** is required,
set by KDPW and the brokerage house:

| Parameter | Indicative value | Meaning |
|---|---|---|
| Initial margin | ~4–6% of contract notional value | Cash blocked when opening a position |
| Maintenance margin (VM) | topped up daily | Variation margin — daily P&L settlement |
| Effective capital leverage | $1 / \\text{margin} \\approx 17$–$25:1$ | E.g. at 5% margin: control of 211 mln PLN notional for ~15 mln PLN capital |
| Full notional exposure | $N \\times K \\times S$ | Still fully felt in P&L — margin is only a settlement mechanism |

**Key difference vs. CFD:** a GPW contract is settled through **KDPW** with an
**external market counterparty** (another exchange participant). Rate changes generate
real cash flows, not an internal broker ledger transfer.

### 2c. Why XTB **Cannot** Hedge with CFDs on Its Own Platform

As an FX CFD issuer, XTB is the counterparty to the client (market-maker /
internalization model). If the company opened a **short CFD EUR/PLN on its own platform**
against an operational **long EUR** exposure, the following would occur:

```
Operational exposure (long EUR)  +  Short CFD on XTB platform (own book)
        ↓                                      ↓
   loss when PLN strengthens          gain on own book (B-book)
   gain when PLN weakens                loss on own book
        ↓                                      ↓
              ≈ NET ZERO transaction at group level
```

CFD hedge P&L **offsets** client-book P&L but **does not create external
protection** — it transfers risk from one P&L line to another within the same entity.
True hedging requires an **external counterparty** (exchange, bank, OTC dealer).

> **Conclusion:** XTB CFD instruments serve retail clients; the company's treasury function
> must use **regulated (GPW)** or **banking (FX forward / NDF)** markets.

### 2d. Proposed Strategy — FEUR/FUSD Contracts on GPW Only

```
┌──────────────────────────────────────────────────────────────┐
│  Operational exposure: +50 mln EUR, +30 mln USD (long FX)   │
└────────────────────────────┬─────────────────────────────────┘
                             │
                             ▼
              ┌──────────────────────────────┐
              │  100% notional — short GPW   │
              │  FEUR + FUSD (KDPW clearing) │
              │  h* ≈ 1.0, quarterly series   │
              └──────────────────────────────┘
```

1. **Position:** short **FEUR** (−50,000 contracts) and **FUSD** (−30,000 contracts) on GPW,
   hedge ratio $h^*$ from spot rate vs. contract price regression.
2. **Rolling:** 5 business days before the third Friday of the delivery month
   (e.g. FEURH26 → FEURM26); roll cost = series price difference (contango/backwardation).
3. **Capital:** margin ~15 mln PLN (5% × notional); reserve for VM and stress margin.
4. **Monitoring:** daily net Delta ≤ 1% of notional; weekly review of hedged VaR.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, t as t_dist
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
np.random.seed(42)

# --- Strategy parameters ---
START, END = '2018-01-01', '2025-12-31'
EXPOSURE_MLN = {'EUR/PLN': 50.0, 'USD/PLN': 30.0}   # mln foreign-currency units
CONTRACT_SIZE = 1_000                              # GPW contract size (EUR/USD)
MARGIN_RATE = 0.05                                 # KDPW margin (~5% of notional)
ALPHA = 0.01                                       # VaR level (99%)

TICKERS = {'EURPLN=X': 'EUR/PLN', 'USDPLN=X': 'USD/PLN', 'GBPPLN=X': 'GBP/PLN'}

raw = yf.download(list(TICKERS.keys()), start=START, end=END, progress=False)['Close']
raw.columns = [TICKERS[c] for c in raw.columns]
spot = raw.dropna()
log_ret = np.log(spot / spot.shift(1)).dropna()

print(f'Data period: {spot.index[0].date()} — {spot.index[-1].date()}')
print(f'Observations: {len(spot)}')
print('\nCurrent spot rates:')
for col in ['EUR/PLN', 'USD/PLN']:
    print(f'  {col}: {spot[col].iloc[-1]:.4f} PLN')
display(spot[['EUR/PLN', 'USD/PLN']].tail(3))

---
## 3. Hedge Ratio and Position Size

For GPW futures contracts, the contract price tracks the spot rate (NBP fixing).
Optimal **hedge ratio** in the minimum-variance sense:

$$
h^* = \frac{\mathrm{Cov}(\Delta S, \Delta F)}{\mathrm{Var}(\Delta F)} \approx \beta_{\Delta S, \Delta F}
$$

Number of contracts (short position):

$$
N = -\,h^* \cdot \frac{Q}{K}
$$

where $Q$ is exposure in foreign currency, $K = 1000$ is the GPW contract size.

In [ ]:
# --- GPW futures price model: F = S + basis (cost-of-carry) ---
# We model basis (F - S) as an AR(1) process (mean-reverting to carry) with innovations
# from t-Student (fat tails) and quarterly jumps when rolling series. This way
# the futures leg moves on its OWN ΔF series, not on spot ΔS — hedging is no longer
# the identity x - x = 0 and reveals real residual risk (basis + roll).
BASIS_PIP        = 1e-4                       # 1 pip = 0,0001 PLN
BASIS_DAILY_STD  = 1.0 * BASIS_PIP           # ~1 pip daily basis change (section 6)
BASIS_PHI        = 0.90                       # basis mean reversion to carry level
ROLL_EVERY       = 63                         # ~quarterly series roll (business days)
ROLL_JUMP_STD    = 3.0 * BASIS_PIP           # price jump between series on roll
BASIS_NU         = 4                          # t-Student degrees of freedom (fat tails)

def build_futures(spot_series, seed):
    """Synthetic contract price = spot + basis (AR(1), fat tails, jumps on roll)."""
    rng = np.random.default_rng(seed)
    n = len(spot_series)
    norm_t = np.sqrt(BASIS_NU / (BASIS_NU - 2))                 # t variance normalization
    sig_eps = BASIS_DAILY_STD * np.sqrt((1 + BASIS_PHI) / 2)    # std(Δbasis) calibration
    eps = rng.standard_t(BASIS_NU, n) / norm_t * sig_eps
    basis = np.zeros(n)
    for i in range(1, n):
        basis[i] = BASIS_PHI * basis[i - 1] + eps[i]
        if i % ROLL_EVERY == 0:                                # jump on series roll
            basis[i] += rng.standard_t(BASIS_NU) / norm_t * ROLL_JUMP_STD
    return spot_series + pd.Series(basis, index=spot_series.index)

futures = {pair: build_futures(spot[pair], seed=100 + i)
           for i, pair in enumerate(['EUR/PLN', 'USD/PLN'])}

def estimate_hedge_ratio(spot_series, fut_series):
    """h* = Cov(ΔS, ΔF)/Var(ΔF) — regression of actual spot vs contract price changes."""
    ds = spot_series.diff().dropna()
    df = fut_series.diff().dropna()
    common = ds.index.intersection(df.index)
    ds, df = ds.loc[common], df.loc[common]
    beta = np.cov(ds, df)[0, 1] / np.var(df)
    rho = np.corrcoef(ds, df)[0, 1]
    return beta, rho

def contracts_needed(exposure_mln, h_star):
    """Number of short GPW contracts (total — contract discreteness)."""
    return -int(round(h_star * exposure_mln * 1e6 / CONTRACT_SIZE))

rows = []
positions = {}
total_margin = 0.0
total_notional_pln = 0.0
for pair in ['EUR/PLN', 'USD/PLN']:
    h, rho = estimate_hedge_ratio(spot[pair], futures[pair])
    n = contracts_needed(EXPOSURE_MLN[pair], h)
    S = spot[pair].iloc[-1]
    notional_pln = abs(n) * CONTRACT_SIZE * S
    margin_pln = notional_pln * MARGIN_RATE
    eff_leverage = 1 / MARGIN_RATE
    positions[pair] = {'h': h, 'n': n}
    total_margin += margin_pln
    total_notional_pln += notional_pln
    ccy = pair.split('/')[0]
    sym = 'FEUR' if ccy == 'EUR' else 'FUSD'
    rows.append({
        'Pair': pair,
        'Exposure [mln]': EXPOSURE_MLN[pair],
        'h* (OLS)': f'{h:.4f}',
        'ρ(ΔS,ΔF)': f'{rho:.4f}',
        f'Short {sym} (GPW)': f'{n:,.0f}',
        'Notional [mln PLN]': f'{notional_pln/1e6:,.1f}',
        'Margin [mln PLN]': f'{margin_pln/1e6:,.1f}',
        'Effective leverage': f'{eff_leverage:.0f}:1',
    })

pos_table = pd.DataFrame(rows)
print('Hedging position — GPW contracts only (100% notional):')
display(pos_table)
print(f'\nTotal initial margin: ~{total_margin/1e6:,.1f} mln PLN')
print(f'Total controlled notional exposure: ~{total_notional_pln/1e6:,.1f} mln PLN')
print(f'Effective capital leverage: ~{total_notional_pln/total_margin:.0f}:1 (at margin {MARGIN_RATE:.0%})')

---
## 4. Sensitivity Measure — Delta ($\Delta$)

**Delta** measures the change in portfolio value per unit change in the spot rate (in PLN):

$$
\Delta_{\text{net}} = \frac{\partial V}{\partial S} = Q - h^* \cdot N \cdot K
$$

For a full hedge $\Delta_{\text{net}} \approx 0$. We also report **Delta in PLN for a 1% rate move**:

$$
\mathrm{PnL}_{1\%} = \Delta_{\text{net}} \cdot S \cdot 0{,}01
$$

In [ ]:
delta_rows = []
for pair in ['EUR/PLN', 'USD/PLN']:
    Q = EXPOSURE_MLN[pair] * 1e6
    h = positions[pair]['h']
    N = positions[pair]['n']
    S = spot[pair].iloc[-1]
    delta_unhedged = Q
    delta_hedged = Q + N * CONTRACT_SIZE  # N negative → reduction
    pnl_1pct_u = delta_unhedged * S * 0.01
    pnl_1pct_h = delta_hedged * S * 0.01
    delta_rows.append({
        'Pair': pair,
        'Δ unhedged [currency units]': f'{delta_unhedged:,.0f}',
        'Δ hedged [currency units]': f'{delta_hedged:,.0f}',
        'Δ reduction': f'{(1 - abs(delta_hedged)/delta_unhedged)*100:.2f} %',
        'PnL at +1% PLN (unhedged)': f'{pnl_1pct_u/1e6:,.2f} mln PLN',
        'PnL at +1% PLN (hedged)': f'{pnl_1pct_h/1e6:,.2f} mln PLN',
    })

delta_table = pd.DataFrame(delta_rows)
print('Delta sensitivity — net exposure vs. hedging strategy:')
display(delta_table)

# Visualization: Delta before and after hedging
fig, ax = plt.subplots(figsize=(8, 4))
pairs = ['EUR/PLN', 'USD/PLN']
x = np.arange(len(pairs))
width = 0.35
d_u = [EXPOSURE_MLN[p]*1e6/1e6 for p in pairs]
d_h = [abs(positions[p]['n']*CONTRACT_SIZE + EXPOSURE_MLN[p]*1e6)/1e6 for p in pairs]
ax.bar(x - width/2, d_u, width, label='|Δ| unhedged [mln units]', color='firebrick', alpha=0.8)
ax.bar(x + width/2, d_h, width, label='|Δ| after hedge [mln units]', color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(pairs)
ax.set_ylabel('Delta magnitude [mln currency units]')
ax.set_title('Delta sensitivity reduction via FEUR/FUSD contracts')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Value at Risk (VaR) of the Hedging Strategy

Daily FX portfolio P&L (in PLN):

$$
\Pi_t = \underbrace{Q_{EUR}\,\Delta S_{EUR,t} + Q_{USD}\,\Delta S_{USD,t}}_{\text{spot exposure}}
+ \underbrace{N_{EUR}\,K\,\Delta S_{EUR,t} + N_{USD}\,K\,\Delta S_{USD,t}}_{\text{FEUR/FUSD contracts}}
$$

We compute **VaR 99%** (1 day) using three methods:
- **Historical (HS)** — empirical 1st percentile of losses,
- **Parametric (variance–covariance)** — assuming normal P&L,
- **t-Student** — fatter tails (consistent with `prez3`/`prez4`).

Additionally **Expected Shortfall (ES)** as a measure coherent with VaR.

In [ ]:
def daily_pnl(exposure_mln, n_contracts, spot_series, fut_series=None):
    """Daily P&L in PLN: exposure on spot ΔS + futures position on ΔF (with basis risk).

    Exposure leg settled on spot change (ΔS), futures leg on change
    in contract price (ΔF = ΔS + Δbasis). Residual hedge P&L ≈ N·K·Δbasis + (Q+N·K)·ΔS.
    """
    dS = spot_series.diff()
    Q = exposure_mln * 1e6
    pnl = Q * dS
    if n_contracts != 0 and fut_series is not None:
        dF = fut_series.diff()
        pnl = pnl + n_contracts * CONTRACT_SIZE * dF
    return pnl

pnl_eur_u = daily_pnl(EXPOSURE_MLN['EUR/PLN'], 0, spot['EUR/PLN'])
pnl_usd_u = daily_pnl(EXPOSURE_MLN['USD/PLN'], 0, spot['USD/PLN'])
pnl_unhedged = pnl_eur_u + pnl_usd_u

pnl_eur_h = daily_pnl(EXPOSURE_MLN['EUR/PLN'], positions['EUR/PLN']['n'], spot['EUR/PLN'], futures['EUR/PLN'])
pnl_usd_h = daily_pnl(EXPOSURE_MLN['USD/PLN'], positions['USD/PLN']['n'], spot['USD/PLN'], futures['USD/PLN'])
pnl_hedged = pnl_eur_h + pnl_usd_h

pnl = pd.DataFrame({'Unhedged': pnl_unhedged, 'Hedged': pnl_hedged}).dropna()
losses = -pnl  # loss as positive value

def var_es(series, alpha=ALPHA):
    x = series.dropna().values
    mu, sig = x.mean(), x.std()
    nu, loc, scale = t_dist.fit(x)
    var_hs = -np.percentile(x, alpha * 100)
    var_norm = -(mu + norm.ppf(alpha) * sig)
    var_t = -(loc + scale * t_dist.ppf(alpha, nu))
    es_hs = -x[x <= np.percentile(x, alpha * 100)].mean()
    return {
        'VaR 99% (HS)': var_hs,
        'VaR 99% (Normal)': var_norm,
        'VaR 99% (t-Student)': var_t,
        'ES 99% (empir.)': es_hs,
        'Daily σ [mln PLN]': sig / 1e6,
    }

var_rows = []
for col in losses.columns:
    m = var_es(-pnl[col])  # losses
    m['Portfolio'] = col
    var_rows.append(m)

var_df = pd.DataFrame(var_rows).set_index('Portfolio')
for c in var_df.columns:
    if 'σ' in c:
        var_df[c] = var_df[c].map(lambda v: f'{v:.3f}')
    else:
        var_df[c] = var_df[c].map(lambda v: f'{v/1e6:,.3f} mln PLN')

print('VaR and ES — comparison of unhedged and FEUR/FUSD-hedged portfolios:')
display(var_df)

# Risk reduction
sig_u = pnl['Unhedged'].std()
sig_h = pnl['Hedged'].std()
var_u = -np.percentile(pnl['Unhedged'], 1)
var_h = -np.percentile(pnl['Hedged'], 1)
print(f'\nDaily σ reduction: {(1 - sig_h/sig_u)*100:.1f}%')
print(f'VaR 99% (HS) reduction: {(1 - var_h/var_u)*100:.1f}%')
print(f'Annual VaR 99% (×√252, HS): unhedged {var_u*np.sqrt(252)/1e6:,.1f} mln PLN  |  hedged {var_h*np.sqrt(252)/1e6:,.2f} mln PLN')

In [ ]:
# --- Charts: P&L, loss distribution, rolling VaR ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('EUR/PLN + USD/PLN hedging strategy — P&L and VaR analysis', fontsize=14, fontweight='bold')

# 1. Cumulative P&L
ax = axes[0, 0]
pnl.cumsum().plot(ax=ax, color=['firebrick', 'steelblue'], lw=0.9)
ax.axhline(0, color='black', lw=0.5)
ax.set_title('Cumulative daily P&L [PLN]')
ax.set_ylabel('PLN')
ax.legend(['Unhedged', 'Hedged (FEUR/FUSD)'])
ax.grid(alpha=0.3)

# 2. Loss histogram — SEPARATE density axes (hedged has ~250x narrower distribution,
#    so on a shared axis the unhedged distribution would flatten to an invisible line).
ax = axes[0, 1]
ax2 = ax.twinx()
bins_u = np.linspace((losses['Unhedged'] / 1e6).min(),
                     (losses['Unhedged'] / 1e6).max(), 60)
ax.hist(losses['Unhedged'] / 1e6, bins=bins_u, alpha=0.55, density=True,
        color='firebrick', label='Unhedged (left axis)')
ax2.hist(losses['Hedged'] / 1e6, bins=80, alpha=0.6, density=True,
         color='steelblue', label='Hedged (right axis)')
var_u_plot = -np.percentile(pnl['Unhedged'], 1) / 1e6
var_h_plot = -np.percentile(pnl['Hedged'], 1) / 1e6
ax.axvline(var_u_plot, color='firebrick', ls='--', lw=1.5,
           label=f'VaR99 unhedged = {var_u_plot:.2f} mln')
ax2.axvline(var_h_plot, color='steelblue', ls='--', lw=1.5,
            label=f'VaR99 hedged = {var_h_plot:.3f} mln')
ax.set_title('Daily loss distribution [mln PLN] — separate density scales')
ax.set_xlabel('Loss [mln PLN]')
ax.set_ylabel('Density — unhedged', color='firebrick')
ax2.set_ylabel('Density — hedged', color='steelblue')
ax.tick_params(axis='y', colors='firebrick')
ax2.tick_params(axis='y', colors='steelblue')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc='upper right')
ax.grid(alpha=0.3)

# 3. Rolling VaR 99% (250-day window)
ax = axes[1, 0]
W = 250
for col, color in zip(pnl.columns, ['firebrick', 'steelblue']):
    roll_var = pnl[col].rolling(W).apply(lambda x: -np.percentile(x, 1), raw=True)
    (roll_var / 1e6).plot(ax=ax, lw=0.9, color=color, label=f'VaR99 {col}')
ax.set_title(f'Rolling VaR 99% ({W}-day window) [mln PLN]')
ax.legend()
ax.grid(alpha=0.3)

# 4. EUR/USD correlation and residual risk
ax = axes[1, 1]
ax.scatter(log_ret['EUR/PLN'], log_ret['USD/PLN'], alpha=0.15, s=8, color='gray')
rho_eu = log_ret['EUR/PLN'].corr(log_ret['USD/PLN'])
ax.set_xlabel('EUR/PLN log return')
ax.set_ylabel('USD/PLN log return')
ax.set_title(f'Cross-correlation of currency pairs (ρ = {rho_eu:.3f})')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Cost Analysis and Residual Risks

| Component | Estimate | Impact on strategy |
|---|---|---|
| **Roll cost** (GPW) | ~0.5–2 pips × notional / quarter | Low vs. potential FX loss |
| **Basis risk** (spot vs. NBP fixing) | σ_basis ≈ 0.5–1 pip/day | Residual risk — acceptable at h* ≈ 1 |
| **Initial margin** (KDPW) | ~5% of notional (~15 mln PLN) | Effective leverage ~20:1, but with daily VM |
| **Variation margin (VM)** | top-up on rate moves | Requires credit line and liquidity reserve |
| **Underestimation of h*** | Re-estimation every 60 days | Rolling refit hedge ratio |
| **CFD na platformie XTB** | — | **Wykluczone** — transakcja net zero w grupie |

**Stress scenario (±3σ PLN move in 1 day):**
- Unhedged: loss of several mln PLN per day.
- With GPW hedge: residual loss < 0.01% of notional (remaining Delta + basis).

In [ ]:
# Stress test: simultaneous EUR and USD move of ±k σ (futures leg tracks spot)
sig_eur = log_ret['EUR/PLN'].std()
sig_usd = log_ret['USD/PLN'].std()
S_eur, S_usd = spot['EUR/PLN'].iloc[-1], spot['USD/PLN'].iloc[-1]

stress_rows = []
for k, label in [(2, '2σ'), (3, '3σ'), (-2, '-2σ (PLN stronger)')]:
    d_eur = k * sig_eur * S_eur
    d_usd = k * sig_usd * S_usd
    loss_u = -(EXPOSURE_MLN['EUR/PLN']*1e6*d_eur + EXPOSURE_MLN['USD/PLN']*1e6*d_usd)
    loss_h = -(positions['EUR/PLN']['n']*CONTRACT_SIZE*d_eur
               + positions['USD/PLN']['n']*CONTRACT_SIZE*d_usd
               + EXPOSURE_MLN['EUR/PLN']*1e6*d_eur + EXPOSURE_MLN['USD/PLN']*1e6*d_usd)
    stress_rows.append({
        'Scenario': label,
        'Δ EUR [PLN]': f'{d_eur:.4f}',
        'Δ USD [PLN]': f'{d_usd:.4f}',
        'Unhedged loss [mln PLN]': f'{loss_u/1e6:,.2f}',
        'Hedged loss [mln PLN]': f'{loss_h/1e6:,.4f}',
    })

stress_df = pd.DataFrame(stress_rows)
print('Scenario test — one-day parallel rate shift (futures tracks spot):')
display(stress_df)

# --- Basis stress: contract price divergence from spot (e.g. 2020 liquidity stress) ---
# Directional hedge offsets spot move but does NOT protect against sudden F vs S divergence.
# Residual loss ≈ N·K·Δbasis (full notional exposure × basis shock).
print('\nBasis stress — sudden contract price vs NBP fixing divergence (with zero spot move):')
basis_rows = []
for pips in [5, 10, 20]:
    db = pips * BASIS_PIP                       # basis shock in PLN (same for both pairs)
    loss_basis = -(positions['EUR/PLN']['n']*CONTRACT_SIZE*db
                   + positions['USD/PLN']['n']*CONTRACT_SIZE*db)
    basis_rows.append({
        'Basis shock': f'{pips} pip',
        'Δbasis [PLN]': f'{db:.4f}',
        'Residual loss [mln PLN]': f'{loss_basis/1e6:,.3f}',
        '% of notional': f'{abs(loss_basis)/total_notional_pln*100:.3f} %',
    })
basis_df = pd.DataFrame(basis_rows)
display(basis_df)
print('Conclusion: even an extreme basis divergence (20 pip/day) yields a loss on the order of hundreds of thousands PLN —')
print('an order of magnitude less than ~4 mln PLN VaR of the unhedged position, but > 0.')

---
## 7. Annual Hedging Costs vs. Potential Unhedged Losses

Hedging on GPW is not free — it carries **explicit costs** (roll, commissions, margin)
and the **opportunity cost** of capital locked in KDPW. We compare them with **empirical
and model-based losses** of the unhedged portfolio on the same data (2018–2025).

**Hedging costs (annual):**

| Component | Formula | Parameter |
|---|---|---|
| Series roll (×4/year) | $\\text{notional} \\times \\text{bps}_{roll} / 10^4$ | 1.5 bps / quarter |
| Brokerage commissions | $N_{contracts} \\times \\text{PLN}_{comm} \\times 2 \\times 4$ | 2.5 PLN / contract / side |
| Opportunity cost of margin | $\\text{margin} \\times r_f$ | $r_f = 5.25\\%$ (52W bond) |

**Unhedged losses (annual):**

| Measure | Description |
|---|---|
| **Annual P&L (empirical)** | Sum of daily FX losses/gains on 50M EUR + 30M USD exposure |
| **Worst year** | Minimum over 8-year sample |
| **Annual VaR 99% (HS)** | 1st percentile of annual P&L |
| **Annual ES 99%** | Average losses in worst 1% of years |
| **VaR 99% (×√252)** | Scaling daily VaR — regulatory benchmark |

In [ ]:
# --- 7. Annual hedging costs vs unhedged losses ---
RF_RATE = 0.0525              # risk-free rate (52W NBP bond, 2025)
ROLLS_PER_YEAR = 4
ROLL_BPS_PER_QUARTER = 1.5    # roll cost (spread between series), in bps of notional
COMM_PLN_PER_CONTRACT = 2.5     # GPW brokerage commission, PLN / contract / side

# Notional and margin (from section 3)
notional_by_pair = {
    pair: abs(positions[pair]['n']) * CONTRACT_SIZE * spot[pair].iloc[-1]
    for pair in ['EUR/PLN', 'USD/PLN']
}
total_notional_pln = sum(notional_by_pair.values())
total_margin_pln = total_notional_pln * MARGIN_RATE
n_contracts_total = sum(abs(positions[p]['n']) for p in positions)

# --- Hedging costs ---
cost_roll = total_notional_pln * (ROLL_BPS_PER_QUARTER / 10_000) * ROLLS_PER_YEAR
cost_comm = n_contracts_total * COMM_PLN_PER_CONTRACT * 2 * ROLLS_PER_YEAR  # close + open
cost_margin = total_margin_pln * RF_RATE
cost_total = cost_roll + cost_comm + cost_margin

cost_breakdown = pd.DataFrame([
    {'Component': 'Series roll (4×/year)', 'Annual cost [mln PLN]': cost_roll/1e6,
     'Share': cost_roll/cost_total},
    {'Component': 'GPW brokerage commissions', 'Annual cost [mln PLN]': cost_comm/1e6,
     'Share': cost_comm/cost_total},
    {'Component': f'Opportunity cost of margin ({RF_RATE:.2%})', 'Annual cost [mln PLN]': cost_margin/1e6,
     'Share': cost_margin/cost_total},
    {'Component': 'TOTAL', 'Annual cost [mln PLN]': cost_total/1e6, 'Share': 1.0},
])
cost_breakdown['Annual cost [mln PLN]'] = cost_breakdown['Annual cost [mln PLN]'].map(lambda v: f'{v:.3f}')
cost_breakdown['Share'] = cost_breakdown['Share'].map(lambda v: f'{v:.1%}')

print('Annual FEUR/FUSD hedge maintenance costs on GPW:')
display(cost_breakdown)

# --- Unhedged losses (empirical) ---
yearly_unhedged = pnl['Unhedged'].groupby(pnl.index.year).sum()
worst_year = yearly_unhedged.min()
best_year = yearly_unhedged.max()
var99_year_hs = -np.percentile(yearly_unhedged.values, 1)
es99_year = -yearly_unhedged[yearly_unhedged <= np.percentile(yearly_unhedged, 1)].mean()
var99_daily_scaled = -np.percentile(pnl['Unhedged'], 1) * np.sqrt(252)

loss_rows = [
    {'Measure': 'Worst year (empirical)', 'Value [mln PLN]': f'{worst_year/1e6:,.2f}',
     'Year': int(yearly_unhedged.idxmin())},
    {'Measure': 'Annual VaR 99% (HS, 8 years)', 'Value [mln PLN]': f'{var99_year_hs/1e6:,.2f}', 'Year': '—'},
    {'Measure': 'Annual ES 99% (empirical)', 'Value [mln PLN]': f'{es99_year/1e6:,.2f}', 'Year': '—'},
    {'Measure': 'Daily VaR 99% × √252', 'Value [mln PLN]': f'{var99_daily_scaled/1e6:,.2f}', 'Year': '—'},
    {'Measure': 'Annual P&L std. dev.', 'Value [mln PLN]': f'{yearly_unhedged.std()/1e6:,.2f}', 'Year': '—'},
]
loss_df = pd.DataFrame(loss_rows)
print('\nPotential unhedged losses:')
display(loss_df)

print('\nAnnual unhedged P&L (mln PLN):')
display((yearly_unhedged/1e6).round(2).to_frame('P&L [mln PLN]'))

# --- Cost–benefit comparison ---
comparison = pd.DataFrame([
    {'Comparison': 'Hedge cost / worst year',
     'Ratio': f'{cost_total/abs(worst_year):.1%}',
     'Interpretation': 'Hedge costs < 9% of worst annual FX loss'},
    {'Comparison': 'Hedge cost / annual VaR 99% (HS)',
     'Ratio': f'{cost_total/var99_year_hs:.1%}',
     'Interpretation': 'Cost << 99% loss threshold'},
    {'Comparison': 'Hedge cost / notional',
     'Ratio': f'{cost_total/total_notional_pln:.2%}',
     'Interpretation': '~0.8% of notional annually'},
    {'Comparison': 'Break-even threshold (FX move)',
     'Ratio': f'{cost_total/total_notional_pln:.2%} annually',
     'Interpretation': 'Hedge "pays off" at > ~0.8% adverse PLN move/year'},
])
print('\nCost–benefit analysis:')
display(comparison)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Annual hedge costs vs unhedged losses', fontsize=13, fontweight='bold')

# Chart 1: cost vs loss measure bars
ax = axes[0]
labels = ['Hedge\ncost', 'VaR99\nannual', 'ES99\nannual', 'Worst\nyear']
values = [cost_total/1e6, var99_year_hs/1e6, es99_year/1e6, abs(worst_year)/1e6]
colors = ['steelblue', 'firebrick', 'darkorange', 'maroon']
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fmt='%.1f mln', padding=3, fontsize=10)
ax.set_ylabel('mln PLN')
ax.set_title('Hedge cost vs potential losses (annual)')
ax.grid(axis='y', alpha=0.3)

# Chart 2: annual unhedged P&L
ax = axes[1]
years = yearly_unhedged.index
vals = yearly_unhedged.values / 1e6
bar_colors = ['firebrick' if v < 0 else 'seagreen' for v in vals]
ax.bar(years, vals, color=bar_colors, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', lw=0.6)
ax.axhline(-cost_total/1e6, color='steelblue', ls='--', lw=1.2,
           label=f'Hedge cost = {cost_total/1e6:.2f} mln PLN/year')
ax.set_xlabel('Year')
ax.set_ylabel('P&L [mln PLN]')
ax.set_title('Annual unhedged FX exposure P&L (2018–2025)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Commission cost sensitivity
print('\nSensitivity — hedge cost at different brokerage commissions:')
for comm in [1.0, 2.5, 5.0, 10.0]:
    c = cost_roll + n_contracts_total * comm * 2 * ROLLS_PER_YEAR + cost_margin
    print(f'  Commission {comm:4.1f} PLN/contract:  cost = {c/1e6:.2f} mln PLN/year  '
          f'({c/abs(worst_year):.1%} of worst year)')

---
## 8. Conclusions and Recommendations

### Strategy
We recommend **hedging exclusively with FEUR and FUSD futures contracts on GPW** (100% notional).
**CFDs on XTB's own platform are excluded** — they do not create external protection because P&L
nets within the group (internalization / B-book). The only sensible tool is a
regulated market with KDPW clearing and an external counterparty.

### Risk measures (results on 2018–2025 data)
| Measure | Before hedge | After hedge | Reduction |
|---|---|---|---|
| **Net Delta** | 80 mln currency units | residual < 0.05% of notional | > 99.9% |
| **Daily P&L σ** | ~1.51 mln PLN | ~0.006 mln PLN | ~99.6% |
| **VaR 99% (1 day, HS)** | ~4.01 mln PLN | ~0.014 mln PLN | ~99.6% |
| **Annual VaR 99% (×√252)** | ~61.7 mln PLN | ~0.25 mln PLN | ~99.6% |

> **Methodological note.** Post-hedge VaR is **non-zero but small** — it reflects residual risk
> explicitly modeled in the contract price: **basis risk** (AR(1) process with fat tails,
> ~1 pip/day) and **jumps on series roll** (quarterly). The futures leg is settled
> on its own price change ΔF = ΔS + Δbasis, so hedging **is not** the identity "ΔS − ΔS = 0".
> Correlation ρ(ΔS,ΔF) ≈ 1.0000 is realistic (same underlying instrument), and residual risk
> stems from basis variance, not directional error.

### Cost vs benefit (section 7)
| | Annually [mln PLN] |
|---|---:|
| **GPW hedge cost** (roll + commissions + margin) | **~2.6** |
| Worst unhedged year (2023) | **−31.1** |
| Annual VaR 99% unhedged | **~30.2** |

Hedge cost represents **~8.5%** of the worst annual loss and **~0.8% of notional** — hedging
"pays off" already at an annual PLN strengthening of about **0.8%** vs. the EUR+USD basket.
At a brokerage commission of 2.5 PLN/contract, transaction cost dominates (~61% of hedge budget).

### Implications for XTB
1. **Market risk management:** daily FX VaR limit after hedging **< 0.5 mln PLN**
   (99%, 1 day); annual hedging budget **~3 mln PLN** vs potential loss **> 30 mln PLN**.
2. **Back-office:** KDPW integration; brokerage commission negotiation (largest cost component).
3. **Consistency with `prez3`–`prez4`:** EUR/USD correlation to PLN > 0.5 — GPW hedging more effective
   than diversification alone; hedge cost << unhedged VaR.
4. **Alternative (banking):** FX forward / NDF — higher OTC spread, but no series rolling.

> **Summary:** **FEUR** and **FUSD** contracts on GPW are the optimal tool
> for hedging XTB S.A.'s EUR/PLN and USD/PLN exposure. At hedge ratio $h^* \approx 1$,
> VaR 99% reduction is **~99.6%** (not 100%), and net Delta falls to a residual level
> (basis + $h^*$ mis-estimation + contract discreteness).